# 行业景气轮动策略复现
## 基于金融数据的量化行业轮动策略研究

本 notebook 展示如何使用多数据源（包括 tushare、baostock、akshare 等）复现行业景气轮动策略。

## 1. 导入必要的库

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'source'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('库导入完成')

## 2. 导入自定义模块

In [ ]:
from source.data_loader import DataLoader
from source.indicators import ProsperityIndicator, IndicatorValidator, ConsensusIndicator
from source.composite_indicator import CompositeIndicatorBuilder, ProsperitySignal, IndustryProsperityCalculator
from source.strategy import ProsperityRotationStrategy, MomentumStrategy, StrategyEvaluator
from source.backtest import BacktestEngine, PortfolioOptimizer
from source.performance import PerformanceAnalyzer, ReturnAttribution
from source.visualization import DataVisualizer
from source.utils import ensure_dir, save_to_csv, log_message

print('自定义模块导入完成')

## 3. 初始化数据加载器

In [ ]:
data_loader = DataLoader()
print('数据加载器初始化完成')

## 4. 获取申万行业列表

In [ ]:
industry_list = data_loader.get_sw_industry_list(level=1)
print(f'获取到 {len(industry_list)} 个申万一级行业')
if len(industry_list) > 0:
    display(industry_list.head(10))

## 5. 获取行业历史价格数据

In [ ]:
start_date = '20200101'
end_date = '20231231'

industry_returns = data_loader.get_sw_industry_historical_batch(
    industry_list['index_code'].tolist()[:10] if len(industry_list) > 10 else industry_list['index_code'].tolist(),
    start_date,
    end_date
)
print(f'获取到 {len(industry_returns)} 条行业收益数据')
if len(industry_returns) > 0:
    display(industry_returns.head())

## 6. 获取行业财务数据

In [ ]:
financial_data = data_loader.get_industry_financial_aggregate(start_date, end_date, reload=True)
print(f'获取到 {len(financial_data)} 条行业财务数据')
if len(financial_data) > 0:
    display(financial_data.head())

## 7. 获取行业一致预期数据

In [ ]:
consensus_data = data_loader.get_consensus_data(start_date, end_date, reload=True)
print(f'获取到 {len(consensus_data)} 条一致预期数据')
if len(consensus_data) > 0:
    display(consensus_data.head())

## 8. 计算景气度指标

In [ ]:
prosperity_indicator = ProsperityIndicator()
industry_indicators = prosperity_indicator.build_industry_indicators(financial_data)
print(f'计算得到 {len(industry_indicators)} 条行业指标数据')
if len(industry_indicators) > 0:
    display(industry_indicators.head())

## 9. 计算一致预期指标

In [ ]:
consensus_indicator = ConsensusIndicator()
consensus_indicators = consensus_indicator.build_consensus_indicators(consensus_data)
print(f'计算得到 {len(consensus_indicators)} 条一致预期指标数据')
if len(consensus_indicators) > 0:
    display(consensus_indicators.head())

## 10. 构建复合景气度指标

In [ ]:
prosperity_calculator = IndustryProsperityCalculator()
prosperity_data = prosperity_calculator.calculate_prosperity_index(
    financial_data,
    consensus_data,
    industry_returns
)
print(f'构建得到 {len(prosperity_data)} 条复合景气度数据')
if len(prosperity_data) > 0:
    display(prosperity_data.head())

## 11. 初始化策略

In [ ]:
strategy = ProsperityRotationStrategy(
    data_loader=data_loader,
    rebalance_freq='Q',
    top_n=5
)

strategy.industry_list = industry_list
strategy.industry_returns = industry_returns
strategy.financial_data = financial_data
strategy.consensus_data = consensus_data
strategy.prosperity_data = prosperity_data
strategy.start_date = start_date
strategy.end_date = end_date

print('策略初始化完成')

## 12. 生成交易信号

In [ ]:
if len(prosperity_data) > 0:
    unique_dates = sorted(prosperity_data['trade_date'].dropna().unique())
    if len(unique_dates) > 0:
        test_date = unique_dates[len(unique_dates)//2]
        signals = strategy.generate_trading_signals(test_date)
        print(f'生成日期 {test_date} 的交易信号')
        if len(signals) > 0:
            display(signals.head(10))

## 13. 运行回测

In [ ]:
backtest_result = strategy.run_backtest()

if backtest_result is not None:
    portfolio_values = backtest_result['portfolio_values']
    signals_df = backtest_result['signals']
    print(f'\n回测完成！')
    print(f'初始资金: {backtest_result["initial_capital"]:,.2f}')
    print(f'最终价值: {backtest_result["final_value"]:,.2f}')
    if len(portfolio_values) > 0:
        display(portfolio_values.head())

## 14. 计算业绩指标

In [ ]:
if len(portfolio_values) > 0:
    returns = portfolio_values['return'].dropna()
    portfolio_value_series = portfolio_values['portfolio_value']
    
    benchmark_returns = strategy.get_benchmark_returns()
    
    performance_analyzer = PerformanceAnalyzer(risk_free_rate=0.03)
    
    if len(benchmark_returns) > 0:
        bench_ret = benchmark_returns['benchmark_return']
        metrics = performance_analyzer.calculate_all_metrics(
            portfolio_value_series, 
            returns, 
            bench_ret
        )
    else:
        metrics = performance_analyzer.calculate_all_metrics(
            portfolio_value_series, 
            returns
        )
    
    print('\n===== 策略业绩指标 =====')
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f'{key}: {value:.4f}')
        else:
            print(f'{key}: {value}')

## 15. 可视化结果

In [ ]:
visualizer = DataVisualizer(figsize=(12, 6))

output_dir = 'output'
ensure_dir(output_dir)

if len(portfolio_values) > 0:
    equity_fig = visualizer.plot_equity_curve(
        portfolio_values['portfolio_value'],
        title='策略权益曲线',
        save_path=os.path.join(output_dir, 'equity_curve.png'),
        show=False
    )
    
    returns_for_plot = portfolio_values['return'].dropna()
    cumret_fig = visualizer.plot_cumulative_returns(
        returns_for_plot,
        title='累计收益',
        save_path=os.path.join(output_dir, 'cumulative_returns.png'),
        show=False
    )
    
    dd_fig = visualizer.plot_drawdown(
        portfolio_values['portfolio_value'],
        title='回撤分析',
        save_path=os.path.join(output_dir, 'drawdown.png'),
        show=False
    )
    
    dist_fig = visualizer.plot_returns_distribution(
        returns_for_plot,
        title='收益分布',
        save_path=os.path.join(output_dir, 'returns_distribution.png'),
        show=False
    )
    
    plt.show()
    print(f'\n图表已保存到 {output_dir} 目录')

## 16. 生成完整回测报告

In [ ]:
if len(portfolio_values) > 0:
    report = performance_analyzer.generate_performance_report(
        portfolio_values['portfolio_value'],
        portfolio_values['return'].dropna()
    )
    print(report)
    
    with open(os.path.join(output_dir, 'performance_report.txt'), 'w', encoding='utf-8') as f:
        f.write(report)
    print(f'\n报告已保存到 {os.path.join(output_dir, "performance_report.txt")}')

## 17. 保存回测结果

In [ ]:
if len(portfolio_values) > 0:
    save_to_csv(portfolio_values, os.path.join(output_dir, 'portfolio_values.csv'))
    
if len(signals_df) > 0:
    save_to_csv(signals_df, os.path.join(output_dir, 'trading_signals.csv'))
    
print(f'回测结果已保存到 {output_dir} 目录')

## 总结

本 notebook 展示了完整的行业景气轮动策略复现流程：

1. **数据获取**: 使用 DataLoader 从 tushare、baostock、akshare 等多数据源获取行业数据
2. **指标计算**: 使用 ProsperityIndicator 和 ConsensusIndicator 计算单个景气度指标
3. **复合指标**: 使用 CompositeIndicatorBuilder 构建复合景气度指标
4. **信号生成**: 根据复合景气度指标生成行业轮动信号
5. **回测验证**: 使用 BacktestEngine 进行策略回测
6. **业绩评估**: 使用 PerformanceAnalyzer 计算各项业绩指标
7. **可视化展示**: 使用 DataVisualizer 生成各种分析图表